# CMS Data Access on NRP

**[website version](https://training.nrp-nautilus.io/cms-hats/5_cms_data.html)** — run cells with **Shift+Enter**.

In this section we will run a CMS-focused Kubernetes Job on NRP, mount an X.509 proxy as a Kubernetes Secret, use `xrdcp` to access a CMS ROOT file, and inspect the file with `uproot`.

Start with an X.509 proxy file you copied from Fermilab LPC or CERN lxplus. If you don't already have one, see [NRP USCMS Analysis Hub](6_analysis_hub.ipynb) for generating a proxy directly from your grid certificate on the hub instead.

## Create an X.509 proxy secret

In [ ]:
export USER=changeme   # ✏️ EDIT to your short name, then Shift+Enter
export X509_PROXY_FILE=/tmp/<x509-proxy-file>   # ✏️ EDIT to your proxy file path


In [ ]:
kubectl create secret generic cms-x509-proxy-${USER} \
  --from-file=proxy="${X509_PROXY_FILE}" \
  -n us-cms \
  --dry-run=client -o yaml | kubectl apply -f -


The Job template mounts this secret and copies it to `/tmp/x509/proxy`, exposed to tools through `X509_USER_PROXY=/tmp/x509/proxy`.

## Build the CMS data image

**🖥️ Terminal step** — Docker build/push. From `workspace/`:

```bash
export IMAGE=ghcr.io/<github-user-or-org>/cms-xrootd-uproot:0.1
docker build --platform linux/amd64 -f code/Dockerfile.cms-data -t "$IMAGE" .
docker push "$IMAGE"
```

If the image is hosted on GHCR, make sure the package is public or create an image pull secret.

In [ ]:
export IMAGE=ghcr.io/<github-user-or-org>/cms-xrootd-uproot:0.1   # ✏️ EDIT to the image you pushed


## Run the CMS data Job

This Job uses the shared training PVC created in the hands-on prep lesson:

In [ ]:
kubectl get pvc -n us-cms cms-nrp-hats-${USER}


`yamls/cms-uproot-job.yaml`:

```yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: cms-uproot-<username>
  namespace: us-cms
spec:
  backoffLimit: 0
  template:
    spec:
      restartPolicy: Never
      securityContext:
        runAsUser: 1000
        runAsGroup: 0
        fsGroup: 0
        fsGroupChangePolicy: OnRootMismatch
      initContainers:
      - name: prepare-x509
        image: busybox:1.36
        command:
        - sh
        - -c
        - cp /secret/proxy /x509/proxy && chmod 600 /x509/proxy
        resources:
          requests:
            cpu: 10m
            memory: 32Mi
          limits:
            cpu: 100m
            memory: 64Mi
        volumeMounts:
        - name: x509-secret
          mountPath: /secret
          readOnly: true
        - name: x509-work
          mountPath: /x509
      containers:
      - name: cms-uproot
        image: <YOUR_IMAGE>
        command: ["python3", "/home/jovyan/work/cms_uproot_example.py"]
        env:
        - name: X509_USER_PROXY
          value: /tmp/x509/proxy
        - name: X509_CERT_DIR
          value: /etc/grid-security/certificates
        - name: LOCAL_ROOT_FILE
          value: /training/cms-data/nanoout_1.root
        - name: SUMMARY_FILE
          value: /training/cms-data/cms_uproot_summary.json
        resources:
          requests:
            cpu: "2"
            memory: 4Gi
          limits:
            cpu: "2"
            memory: 4Gi
        volumeMounts:
        - name: x509-work
          mountPath: /tmp/x509
          readOnly: true
        - name: training-storage
          mountPath: /training
      volumes:
      - name: x509-secret
        secret:
          secretName: cms-x509-proxy-<username>
      - name: x509-work
        emptyDir: {}
      - name: training-storage
        persistentVolumeClaim:
          claimName: cms-nrp-hats-<username>
```

Create a temporary copy of the Job manifest and replace the placeholders:

In [ ]:
cd ~/cms-hats/workspace
cp yamls/cms-uproot-job.yaml /tmp/cms-uproot-${USER}.yaml
perl -pi -e 's/<username>/$ENV{USER}/g; s|<YOUR_IMAGE>|$ENV{IMAGE}|g' /tmp/cms-uproot-${USER}.yaml


In [ ]:
kubectl delete job -n us-cms cms-uproot-${USER} --ignore-not-found


In [ ]:
kubectl apply -n us-cms -f /tmp/cms-uproot-${USER}.yaml


In [ ]:
kubectl get jobs,pods -n us-cms


**🖥️ Terminal step** — follows the log stream:

```bash
kubectl logs -n us-cms job/cms-uproot-${USER} -f
```

In [ ]:
kubectl get job -n us-cms cms-uproot-${USER}


## Experimental: Jupyter pod

The NRP documentation generally recommends JupyterHub for quick interactive work, but also documents running your own Jupyter pod when you need a custom container image: [ML/Jupyter pod](https://nrp.ai/documentation/userdocs/jupyter/jupyter-pod/). This whole section is optional — the batch Job above is the preferred path for a reproducible run.

`yamls/cms-jupyter-pod.yaml`:

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: cms-jupyter-<username>
  namespace: us-cms
spec:
  restartPolicy: Never
  securityContext:
    runAsUser: 1000
    runAsGroup: 0
    fsGroup: 0
    fsGroupChangePolicy: OnRootMismatch
  initContainers:
  - name: prepare-x509
    image: busybox:1.36
    command:
    - sh
    - -c
    - cp /secret/proxy /x509/proxy && chmod 600 /x509/proxy
    resources:
      requests:
        cpu: 10m
        memory: 32Mi
      limits:
        cpu: 100m
        memory: 64Mi
    volumeMounts:
    - name: x509-secret
      mountPath: /secret
      readOnly: true
    - name: x509-work
      mountPath: /x509
  containers:
  - name: cms-jupyter
    image: <YOUR_IMAGE>
    ports:
    - containerPort: 8888
    env:
    - name: X509_USER_PROXY
      value: /tmp/x509/proxy
    - name: X509_CERT_DIR
      value: /etc/grid-security/certificates
    - name: LOCAL_ROOT_FILE
      value: /training/cms-data/notebook/nanoout_1.root
    - name: SUMMARY_FILE
      value: /training/cms-data/notebook/cms_uproot_summary.json
    resources:
      requests:
        cpu: "2"
        memory: 4Gi
      limits:
        cpu: "2"
        memory: 4Gi
    volumeMounts:
    - name: x509-work
      mountPath: /tmp/x509
      readOnly: true
    - name: training-storage
      mountPath: /training
  volumes:
  - name: x509-secret
    secret:
      secretName: cms-x509-proxy-<username>
  - name: x509-work
    emptyDir: {}
  - name: training-storage
    persistentVolumeClaim:
      claimName: cms-nrp-hats-<username>
```

**🖥️ Terminal step** — port-forward and browser interaction don't fit the notebook-cell model. From a terminal:

```bash
cd ~/cms-hats/workspace
cp yamls/cms-jupyter-pod.yaml /tmp/cms-jupyter-${USER}.yaml
perl -pi -e 's/<username>/$ENV{USER}/g; s|<YOUR_IMAGE>|$ENV{IMAGE}|g' /tmp/cms-jupyter-${USER}.yaml
kubectl delete pod -n us-cms cms-jupyter-${USER} --ignore-not-found
kubectl apply -n us-cms -f /tmp/cms-jupyter-${USER}.yaml
kubectl wait -n us-cms --for=condition=Ready pod/cms-jupyter-${USER} --timeout=10m
kubectl port-forward -n us-cms pod/cms-jupyter-${USER} 8888:8888
```

In another terminal, get the token with `kubectl logs -n us-cms pod/cms-jupyter-${USER}`, then open `http://localhost:8888` and open `cms_uproot_example.ipynb`.

## Clean up

In [ ]:
kubectl delete job -n us-cms cms-uproot-${USER}


If you started the experimental Jupyter pod, stop the port-forward with Ctrl-C, then:

```bash
kubectl delete pod -n us-cms cms-jupyter-${USER}
```

In [ ]:
kubectl delete secret -n us-cms cms-x509-proxy-${USER}   # only if you don't plan to reuse it


---

## ✅ Check your work

Verifies the state of your resources on the cluster — rerun any time.

In [ ]:
bash check.sh 5
